# Cross-Dataset Generalization and Failure Detection with EfficientNet-B0

This notebook evaluates EfficientNet-B0 for four-class brain MRI classification across the BDNeuro and PMRAM datasets. Experiments are conducted in both cross-dataset directions, followed by failure detection using MSR, MLS, and MC-Dropout.

In [2]:
# ============================================================
# EfficientNet-B0
# Complete Data Pipeline + Model Setup
# Same protocol as ResNet18
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import pandas as pd
import numpy as np
import time

from PIL import Image

from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    roc_curve
)


# ============================================================
# 1. Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# 2. Load final dataset metadata
# EXACTLY SAME CSV AS RESNET18
# ============================================================

csv_path = "final_four_class_clean_dataset.csv"

df = pd.read_csv(csv_path)

print("\nDataset size:", len(df))

print("\nDataset counts:")
print(df["dataset"].value_counts())


# ============================================================
# 3. Separate the two datasets
# ============================================================

bdneuro_df = df[
    df["dataset"] == "BDNeuro"
].copy()

pmram_df = df[
    df["dataset"] == "PMRAM"
].copy()

print("\nBDNeuro:", len(bdneuro_df))
print("PMRAM:", len(pmram_df))


# ============================================================
# 4. BDNeuro original split
# ============================================================

bd_train_df = bdneuro_df[
    bdneuro_df["original_split"] == "train"
].copy()

bd_val_df = bdneuro_df[
    bdneuro_df["original_split"] == "val"
].copy()

bd_test_df = bdneuro_df[
    bdneuro_df["original_split"] == "test"
].copy()


# ============================================================
# 5. PMRAM split
# EXACTLY SAME AS RESNET18
# 70% train / 15% validation / 15% test
# ============================================================

pm_train_df, pm_temp_df = train_test_split(
    pmram_df,
    test_size=0.30,
    stratify=pmram_df["class_label"],
    random_state=42
)

pm_val_df, pm_test_df = train_test_split(
    pm_temp_df,
    test_size=0.50,
    stratify=pm_temp_df["class_label"],
    random_state=42
)


print("\n--- Dataset Splits ---")

print("BDNeuro Train:", len(bd_train_df))
print("BDNeuro Validation:", len(bd_val_df))
print("BDNeuro Test:", len(bd_test_df))

print()

print("PMRAM Train:", len(pm_train_df))
print("PMRAM Validation:", len(pm_val_df))
print("PMRAM Test:", len(pm_test_df))


# ============================================================
# 6. Image preprocessing
# EXACTLY SAME AS RESNET18
# ============================================================

image_size = 224
batch_size = 32

train_transform = transforms.Compose([

    transforms.Resize(
        (image_size, image_size)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


eval_transform = transforms.Compose([

    transforms.Resize(
        (image_size, image_size)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# 7. Dataset class
# IMAGE PATH IS READ FROM THE CSV
# ============================================================

class BrainMRIDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

        self.class_to_idx = {
            "glioma": 0,
            "meningioma": 1,
            "no_tumor": 2,
            "pituitary": 3
        }


    def __len__(self):

        return len(
            self.dataframe
        )


    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        # Image address comes directly
        # from the filepath column in CSV

        image = Image.open(
            row["filepath"]
        ).convert("RGB")

        label = self.class_to_idx[
            row["class_label"]
        ]

        if self.transform:

            image = self.transform(
                image
            )

        return image, label


# ============================================================
# 8. Create datasets
# ============================================================

bd_train_dataset = BrainMRIDataset(
    bd_train_df,
    transform=train_transform
)

bd_val_dataset = BrainMRIDataset(
    bd_val_df,
    transform=eval_transform
)

bd_test_dataset = BrainMRIDataset(
    bd_test_df,
    transform=eval_transform
)


pm_train_dataset = BrainMRIDataset(
    pm_train_df,
    transform=train_transform
)

pm_val_dataset = BrainMRIDataset(
    pm_val_df,
    transform=eval_transform
)

pm_test_dataset = BrainMRIDataset(
    pm_test_df,
    transform=eval_transform
)


# Complete external datasets

pm_external_dataset = BrainMRIDataset(
    pmram_df,
    transform=eval_transform
)

bd_external_dataset = BrainMRIDataset(
    bdneuro_df,
    transform=eval_transform
)


# ============================================================
# 9. DataLoaders
# ============================================================

bd_train_loader = DataLoader(
    bd_train_dataset,
    batch_size=batch_size,
    shuffle=True
)

bd_val_loader = DataLoader(
    bd_val_dataset,
    batch_size=batch_size,
    shuffle=False
)

bd_test_loader = DataLoader(
    bd_test_dataset,
    batch_size=batch_size,
    shuffle=False
)


pm_train_loader = DataLoader(
    pm_train_dataset,
    batch_size=batch_size,
    shuffle=True
)

pm_val_loader = DataLoader(
    pm_val_dataset,
    batch_size=batch_size,
    shuffle=False
)

pm_test_loader = DataLoader(
    pm_test_dataset,
    batch_size=batch_size,
    shuffle=False
)


pm_external_loader = DataLoader(
    pm_external_dataset,
    batch_size=batch_size,
    shuffle=False
)

bd_external_loader = DataLoader(
    bd_external_dataset,
    batch_size=batch_size,
    shuffle=False
)


print("\nAll DataLoaders created successfully.")


# ============================================================
# 10. Quick image-path test
# Make sure images can actually be opened
# ============================================================

test_image, test_label = bd_train_dataset[0]

print("\nImage path test successful.")
print("Image tensor shape:", test_image.shape)
print("Test label:", test_label)


# ============================================================
# 11. EfficientNet-B0
# ImageNet pretrained
# ============================================================

model_bd_eff = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

in_features = (
    model_bd_eff.classifier[1].in_features
)

model_bd_eff.classifier[1] = nn.Linear(
    in_features,
    4
)

model_bd_eff = model_bd_eff.to(
    device
)


# ============================================================
# 12. Full fine-tuning
# ============================================================

for parameter in model_bd_eff.parameters():

    parameter.requires_grad = True


total_parameters = sum(
    p.numel()
    for p in model_bd_eff.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model_bd_eff.parameters()
    if p.requires_grad
)


print("\nEfficientNet-B0 created successfully.")

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)


# ============================================================
# 13. Training configuration
# SAME AS RESNET18
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer_bd_eff = optim.Adam(
    model_bd_eff.parameters(),
    lr=0.0001
)


print("\nLoss: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 0.0001")
print("Batch size:", batch_size)
print("Image size:", image_size)

print("\n======================================")
print("EFFICIENTNET-B0 SETUP COMPLETE")
print("======================================")


Device: cpu

Dataset size: 7351

Dataset counts:
dataset
BDNeuro    5941
PMRAM      1410
Name: count, dtype: int64

BDNeuro: 5941
PMRAM: 1410

--- Dataset Splits ---
BDNeuro Train: 4160
BDNeuro Validation: 892
BDNeuro Test: 889

PMRAM Train: 987
PMRAM Validation: 211
PMRAM Test: 212

All DataLoaders created successfully.

Image path test successful.
Image tensor shape: torch.Size([3, 224, 224])
Test label: 0
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\USER/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|█████████████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:06<00:00, 3.20MB/s]



EfficientNet-B0 created successfully.
Total parameters: 4,012,672
Trainable parameters: 4,012,672

Loss: CrossEntropyLoss
Optimizer: Adam
Learning rate: 0.0001
Batch size: 32
Image size: 224

EFFICIENTNET-B0 SETUP COMPLETE


## 1. Cross-Dataset Classification

### BDNeuro → PMRAM

EfficientNet-B0 is trained on BDNeuro and evaluated on both the BDNeuro internal test set and the complete PMRAM dataset as an external test set.


In [4]:
# ============================================================
# EfficientNet-B0 Training
# Experiment 1: BDNeuro -> PMRAM
# 5 Epochs + Best Validation Model
# ============================================================

import copy
import time

def train_best_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs=5
):

    best_val_accuracy = 0.0
    best_epoch = 0
    best_model_weights = copy.deepcopy(
        model.state_dict()
    )

    total_start_time = time.time()

    for epoch in range(num_epochs):

        epoch_start_time = time.time()

        # ====================================================
        # TRAINING
        # ====================================================

        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            train_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            train_correct += (
                predictions == labels
            ).sum().item()

            train_total += labels.size(0)


        train_loss = (
            train_loss / train_total
        )

        train_accuracy = (
            train_correct / train_total
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

                val_loss += (
                    loss.item() * images.size(0)
                )

                predictions = outputs.argmax(
                    dim=1
                )

                val_correct += (
                    predictions == labels
                ).sum().item()

                val_total += labels.size(0)


        val_loss = (
            val_loss / val_total
        )

        val_accuracy = (
            val_correct / val_total
        )


        # ====================================================
        # SAVE BEST MODEL IN MEMORY
        # ====================================================

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy

            best_epoch = epoch + 1

            best_model_weights = copy.deepcopy(
                model.state_dict()
            )


        # ====================================================
        # TIME
        # ====================================================

        epoch_minutes = (
            time.time() - epoch_start_time
        ) / 60


        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f} | "
            f"Time: {epoch_minutes:.1f} min"
        )


    # ========================================================
    # LOAD BEST VALIDATION MODEL
    # ========================================================

    model.load_state_dict(
        best_model_weights
    )


    total_minutes = (
        time.time() - total_start_time
    ) / 60


    print("\n======================================")
    print("TRAINING COMPLETE")
    print("Best Epoch:", best_epoch)
    print(
        "Best Validation Accuracy:",
        f"{best_val_accuracy:.4f}"
    )
    print(
        "Total Training Time:",
        f"{total_minutes:.1f} minutes"
    )
    print("======================================")


    return model


# ============================================================
# Train EfficientNet-B0
# BDNeuro -> PMRAM
# ============================================================

model_bd_eff = train_best_model(
    model=model_bd_eff,
    train_loader=bd_train_loader,
    val_loader=bd_val_loader,
    criterion=criterion,
    optimizer=optimizer_bd_eff,
    device=device,
    num_epochs=5
)


# ============================================================
# Save BEST model permanently
# ============================================================

torch.save(
    model_bd_eff.state_dict(),
    "efficientnet_b0_bdneuro_best.pth"
)

print(
    "\nBest EfficientNet-B0 model saved as:"
)
print(
    "efficientnet_b0_bdneuro_best.pth"
)


Epoch 1/5 | Train Loss: 0.6433 | Train Acc: 0.7762 | Val Loss: 0.6941 | Val Acc: 0.9025 | Time: 16.8 min
Epoch 2/5 | Train Loss: 0.2457 | Train Acc: 0.9178 | Val Loss: 0.1747 | Val Acc: 0.9439 | Time: 16.3 min
Epoch 3/5 | Train Loss: 0.1396 | Train Acc: 0.9510 | Val Loss: 0.1232 | Val Acc: 0.9552 | Time: 16.3 min
Epoch 4/5 | Train Loss: 0.0985 | Train Acc: 0.9704 | Val Loss: 0.0949 | Val Acc: 0.9630 | Time: 16.7 min
Epoch 5/5 | Train Loss: 0.0807 | Train Acc: 0.9736 | Val Loss: 0.0847 | Val Acc: 0.9720 | Time: 16.4 min

TRAINING COMPLETE
Best Epoch: 5
Best Validation Accuracy: 0.9720
Total Training Time: 82.6 minutes

Best EfficientNet-B0 model saved as:
efficientnet_b0_bdneuro_best.pth


In [5]:
# ============================================================
# EfficientNet-B0 Evaluation
# BDNeuro Internal + PMRAM External
# ============================================================

def evaluate_model(model, data_loader, device):

    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            predictions = outputs.argmax(dim=1)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    report = classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Glioma",
            "Meningioma",
            "No Tumor",
            "Pituitary"
        ],
        digits=4
    )

    return accuracy, report


# ============================================================
# Internal: BDNeuro
# ============================================================

bd_eff_test_accuracy, bd_eff_test_report = evaluate_model(
    model_bd_eff,
    bd_test_loader,
    device
)

print("BDNeuro Internal Test Accuracy:")
print(bd_eff_test_accuracy)

print("\nBDNeuro Classification Report:")
print(bd_eff_test_report)


# ============================================================
# External: PMRAM
# ============================================================

pm_eff_external_accuracy, pm_eff_external_report = evaluate_model(
    model_bd_eff,
    pm_external_loader,
    device
)

print("\n======================================")

print("PMRAM External Test Accuracy:")
print(pm_eff_external_accuracy)

print("\nPMRAM External Classification Report:")
print(pm_eff_external_report)


BDNeuro Internal Test Accuracy:
0.9786276715410573

BDNeuro Classification Report:
              precision    recall  f1-score   support

      Glioma     0.9846    0.9756    0.9801       328
  Meningioma     0.9552    0.9638    0.9595       221
    No Tumor     0.9833    1.0000    0.9916       118
   Pituitary     0.9910    0.9865    0.9887       222

    accuracy                         0.9786       889
   macro avg     0.9785    0.9815    0.9800       889
weighted avg     0.9787    0.9786    0.9786       889


PMRAM External Test Accuracy:
0.9595744680851064

PMRAM External Classification Report:
              precision    recall  f1-score   support

      Glioma     0.9730    0.9757    0.9744       370
  Meningioma     0.9696    0.8777    0.9213       327
    No Tumor     0.9577    0.9971    0.9770       341
   Pituitary     0.9407    0.9812    0.9605       372

    accuracy                         0.9596      1410
   macro avg     0.9603    0.9579    0.9583      1410
weighted avg 

### PMRAM → BDNeuro

The experiment is repeated in the reverse direction by training on PMRAM and evaluating on BDNeuro. This allows the directional asymmetry of cross-dataset generalization to be examined.

In [6]:
# ============================================================
# Experiment 2: PMRAM -> BDNeuro
# EfficientNet-B0
# ============================================================

import torch
import torch.nn as nn
from torchvision import models
import time
import copy

# ------------------------------------------------------------
# 1. Create a NEW EfficientNet-B0
# ------------------------------------------------------------

model_pm_eff = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

in_features = model_pm_eff.classifier[1].in_features

model_pm_eff.classifier[1] = nn.Linear(
    in_features,
    4
)

model_pm_eff = model_pm_eff.to(device)

# Full fine-tuning
for parameter in model_pm_eff.parameters():
    parameter.requires_grad = True


# ------------------------------------------------------------
# 2. Loss + Optimizer
# Same learning rate as previous experiment
# ------------------------------------------------------------

criterion_pm_eff = nn.CrossEntropyLoss()

optimizer_pm_eff = torch.optim.Adam(
    model_pm_eff.parameters(),
    lr=0.0001
)


# ------------------------------------------------------------
# 3. Training - keep best validation model
# ------------------------------------------------------------

num_epochs = 5

best_val_accuracy = 0.0
best_model_weights = copy.deepcopy(
    model_pm_eff.state_dict()
)

training_start = time.time()

print("Training EfficientNet-B0: PMRAM -> BDNeuro")
print("=" * 60)


for epoch in range(num_epochs):

    epoch_start = time.time()

    # ========================
    # TRAINING
    # ========================

    model_pm_eff.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in pm_train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_pm_eff.zero_grad()

        outputs = model_pm_eff(images)

        loss = criterion_pm_eff(
            outputs,
            labels
        )

        loss.backward()

        optimizer_pm_eff.step()

        train_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(dim=1)

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)


    train_loss = train_loss / train_total

    train_accuracy = (
        train_correct / train_total
    )


    # ========================
    # VALIDATION
    # ========================

    model_pm_eff.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in pm_val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_pm_eff(images)

            loss = criterion_pm_eff(
                outputs,
                labels
            )

            val_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)


    val_loss = val_loss / val_total

    val_accuracy = (
        val_correct / val_total
    )


    # ========================
    # SAVE BEST MODEL
    # ========================

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        best_model_weights = copy.deepcopy(
            model_pm_eff.state_dict()
        )


    epoch_minutes = (
        time.time() - epoch_start
    ) / 60


    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Time: {epoch_minutes:.1f} min"
    )


# ------------------------------------------------------------
# 4. Restore BEST validation model
# ------------------------------------------------------------

model_pm_eff.load_state_dict(
    best_model_weights
)


total_minutes = (
    time.time() - training_start
) / 60


print("\n" + "=" * 60)

print("TRAINING COMPLETE")

print(
    f"Best Validation Accuracy: "
    f"{best_val_accuracy:.4f}"
)

print(
    f"Total Training Time: "
    f"{total_minutes:.1f} minutes"
)


# ------------------------------------------------------------
# 5. Save model
# ------------------------------------------------------------

torch.save(
    model_pm_eff.state_dict(),
    "efficientnet_b0_pmram_best.pth"
)

print(
    "\nBest EfficientNet-B0 model saved as:"
)

print(
    "efficientnet_b0_pmram_best.pth"
)


Training EfficientNet-B0: PMRAM -> BDNeuro
Epoch 1/5 | Train Loss: 0.9952 | Train Acc: 0.6940 | Val Loss: 0.7203 | Val Acc: 0.7915 | Time: 5.2 min
Epoch 2/5 | Train Loss: 0.4259 | Train Acc: 0.8946 | Val Loss: 0.2765 | Val Acc: 0.9052 | Time: 3.7 min
Epoch 3/5 | Train Loss: 0.2260 | Train Acc: 0.9240 | Val Loss: 0.1665 | Val Acc: 0.9479 | Time: 3.7 min
Epoch 4/5 | Train Loss: 0.1404 | Train Acc: 0.9574 | Val Loss: 0.1592 | Val Acc: 0.9479 | Time: 3.6 min
Epoch 5/5 | Train Loss: 0.1038 | Train Acc: 0.9696 | Val Loss: 0.0958 | Val Acc: 0.9810 | Time: 3.6 min

TRAINING COMPLETE
Best Validation Accuracy: 0.9810
Total Training Time: 19.8 minutes

Best EfficientNet-B0 model saved as:
efficientnet_b0_pmram_best.pth


In [7]:
# ============================================================
# EfficientNet-B0 Evaluation
# PMRAM -> BDNeuro
# ============================================================

from sklearn.metrics import accuracy_score, classification_report

def evaluate_eff_model(model, data_loader, device):

    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    report = classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Glioma",
            "Meningioma",
            "No Tumor",
            "Pituitary"
        ],
        digits=4
    )

    return accuracy, report


# ============================================================
# 1. PMRAM INTERNAL TEST
# ============================================================

pm_eff_test_accuracy, pm_eff_test_report = evaluate_eff_model(
    model_pm_eff,
    pm_test_loader,
    device
)

print("=" * 60)
print("PMRAM INTERNAL TEST")
print("=" * 60)

print(
    "\nPMRAM Internal Test Accuracy:",
    pm_eff_test_accuracy
)

print("\nPMRAM Classification Report:")
print(pm_eff_test_report)


# ============================================================
# 2. BDNEuro EXTERNAL TEST
# ============================================================

bd_eff_external_accuracy, bd_eff_external_report = evaluate_eff_model(
    model_pm_eff,
    bd_external_loader,
    device
)

print("\n" + "=" * 60)
print("BDNeuro EXTERNAL TEST")
print("=" * 60)

print(
    "\nBDNeuro External Test Accuracy:",
    bd_eff_external_accuracy
)

print("\nBDNeuro External Classification Report:")
print(bd_eff_external_report)


PMRAM INTERNAL TEST

PMRAM Internal Test Accuracy: 0.9622641509433962

PMRAM Classification Report:
              precision    recall  f1-score   support

      Glioma     0.9636    0.9464    0.9550        56
  Meningioma     0.9565    0.8980    0.9263        49
    No Tumor     0.9273    1.0000    0.9623        51
   Pituitary     1.0000    1.0000    1.0000        56

    accuracy                         0.9623       212
   macro avg     0.9619    0.9611    0.9609       212
weighted avg     0.9628    0.9623    0.9620       212


BDNeuro EXTERNAL TEST

BDNeuro External Test Accuracy: 0.6628513718229254

BDNeuro External Classification Report:
              precision    recall  f1-score   support

      Glioma     0.7419    0.8357    0.7860      2191
  Meningioma     0.8527    0.1685    0.2814      1478
    No Tumor     0.4321    0.9822    0.6002       787
   Pituitary     0.7795    0.7306    0.7543      1485

    accuracy                         0.6629      5941
   macro avg     0.7015

## 2. Failure Detection with MSR and MLS

Prediction failures are evaluated using Maximum Softmax Response (MSR) and Maximum Logit Score (MLS). Failure-detection thresholds are selected only from the source validation set and then applied unchanged to the external dataset.





In [8]:
# ============================================================
# EfficientNet-B0 Failure Detection
# MSR + MLS
# Both Cross-Dataset Directions
# ============================================================

import torch
import torch.nn.functional as F
import pandas as pd

from sklearn.metrics import roc_auc_score, roc_curve


# ============================================================
# 1. Collect MSR + MLS together
# ============================================================

def collect_eff_confidence_results(model, data_loader, device):

    model.eval()
    results = []

    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            # Softmax probabilities
            probabilities = F.softmax(outputs, dim=1)

            # MSR
            msr, predictions = probabilities.max(dim=1)

            # MLS
            mls, _ = outputs.max(dim=1)

            for label, prediction, msr_score, mls_score in zip(
                labels.cpu(),
                predictions.cpu(),
                msr.cpu(),
                mls.cpu()
            ):

                results.append({
                    "true_label": label.item(),
                    "predicted_label": prediction.item(),
                    "msr": msr_score.item(),
                    "mls": mls_score.item(),
                    "correct": int(
                        label.item() == prediction.item()
                    )
                })

    return pd.DataFrame(results)


# ============================================================
# 2. AUROC-F functions
# ============================================================

def calculate_msr_auroc(results):

    failure_labels = 1 - results["correct"]
    failure_scores = 1 - results["msr"]

    return roc_auc_score(
        failure_labels,
        failure_scores
    )


def calculate_mls_auroc(results):

    failure_labels = 1 - results["correct"]
    failure_scores = -results["mls"]

    return roc_auc_score(
        failure_labels,
        failure_scores
    )


# ============================================================
# 3. Threshold functions
# Source VALIDATION only
# ============================================================

def find_msr_threshold(results):

    failure_labels = 1 - results["correct"]
    failure_scores = 1 - results["msr"]

    fpr, tpr, thresholds = roc_curve(
        failure_labels,
        failure_scores
    )

    best_index = (tpr - fpr).argmax()

    failure_threshold = thresholds[best_index]

    return 1 - failure_threshold


def find_mls_threshold(results):

    failure_labels = 1 - results["correct"]
    failure_scores = -results["mls"]

    fpr, tpr, thresholds = roc_curve(
        failure_labels,
        failure_scores
    )

    best_index = (tpr - fpr).argmax()

    failure_threshold = thresholds[best_index]

    return -failure_threshold


# ============================================================
# 4. Threshold evaluation
# ============================================================

def evaluate_threshold(results, score_name, threshold):

    results = results.copy()

    results["rejected"] = (
        results[score_name] < threshold
    )

    total_errors = (
        results["correct"] == 0
    ).sum()

    caught_errors = (
        (results["correct"] == 0) &
        results["rejected"]
    ).sum()

    silent_failures = (
        total_errors - caught_errors
    )

    accepted = (
        ~results["rejected"]
    ).sum()

    coverage = (
        accepted / len(results)
    )

    if total_errors > 0:
        detection_rate = (
            caught_errors / total_errors
        )
    else:
        detection_rate = float("nan")

    return (
        detection_rate,
        coverage,
        silent_failures
    )


# ============================================================
# 5. BDNeuro -> PMRAM
# ============================================================

print("Processing BDNeuro -> PMRAM...")

bd_eff_val_conf = collect_eff_confidence_results(
    model_bd_eff,
    bd_val_loader,
    device
)

bd_eff_test_conf = collect_eff_confidence_results(
    model_bd_eff,
    bd_test_loader,
    device
)

pm_eff_external_conf = collect_eff_confidence_results(
    model_bd_eff,
    pm_external_loader,
    device
)


# AUROC-F

bd_eff_internal_msr_auroc = calculate_msr_auroc(
    bd_eff_test_conf
)

pm_eff_external_msr_auroc = calculate_msr_auroc(
    pm_eff_external_conf
)

bd_eff_internal_mls_auroc = calculate_mls_auroc(
    bd_eff_test_conf
)

pm_eff_external_mls_auroc = calculate_mls_auroc(
    pm_eff_external_conf
)


# Source thresholds

bd_eff_msr_threshold = find_msr_threshold(
    bd_eff_val_conf
)

bd_eff_mls_threshold = find_mls_threshold(
    bd_eff_val_conf
)


# External threshold performance

d1_eff_msr_detection, \
d1_eff_msr_coverage, \
d1_eff_msr_silent = evaluate_threshold(
    pm_eff_external_conf,
    "msr",
    bd_eff_msr_threshold
)

d1_eff_mls_detection, \
d1_eff_mls_coverage, \
d1_eff_mls_silent = evaluate_threshold(
    pm_eff_external_conf,
    "mls",
    bd_eff_mls_threshold
)


# ============================================================
# 6. PMRAM -> BDNeuro
# ============================================================

print("Processing PMRAM -> BDNeuro...")

pm_eff_val_conf = collect_eff_confidence_results(
    model_pm_eff,
    pm_val_loader,
    device
)

pm_eff_test_conf = collect_eff_confidence_results(
    model_pm_eff,
    pm_test_loader,
    device
)

bd_eff_external_conf = collect_eff_confidence_results(
    model_pm_eff,
    bd_external_loader,
    device
)


# AUROC-F

pm_eff_internal_msr_auroc = calculate_msr_auroc(
    pm_eff_test_conf
)

bd_eff_external_msr_auroc = calculate_msr_auroc(
    bd_eff_external_conf
)

pm_eff_internal_mls_auroc = calculate_mls_auroc(
    pm_eff_test_conf
)

bd_eff_external_mls_auroc = calculate_mls_auroc(
    bd_eff_external_conf
)


# Source thresholds

pm_eff_msr_threshold = find_msr_threshold(
    pm_eff_val_conf
)

pm_eff_mls_threshold = find_mls_threshold(
    pm_eff_val_conf
)


# External threshold performance

d2_eff_msr_detection, \
d2_eff_msr_coverage, \
d2_eff_msr_silent = evaluate_threshold(
    bd_eff_external_conf,
    "msr",
    pm_eff_msr_threshold
)

d2_eff_mls_detection, \
d2_eff_mls_coverage, \
d2_eff_mls_silent = evaluate_threshold(
    bd_eff_external_conf,
    "mls",
    pm_eff_mls_threshold
)


# ============================================================
# 7. FINAL TABLE
# ============================================================

efficientnet_failure_summary = pd.DataFrame({

    "Direction": [
        "BDNeuro -> PMRAM",
        "BDNeuro -> PMRAM",
        "PMRAM -> BDNeuro",
        "PMRAM -> BDNeuro"
    ],

    "Method": [
        "MSR",
        "MLS",
        "MSR",
        "MLS"
    ],

    "Internal AUROC-F": [
        bd_eff_internal_msr_auroc,
        bd_eff_internal_mls_auroc,
        pm_eff_internal_msr_auroc,
        pm_eff_internal_mls_auroc
    ],

    "External AUROC-F": [
        pm_eff_external_msr_auroc,
        pm_eff_external_mls_auroc,
        bd_eff_external_msr_auroc,
        bd_eff_external_mls_auroc
    ],

    "Source Threshold": [
        bd_eff_msr_threshold,
        bd_eff_mls_threshold,
        pm_eff_msr_threshold,
        pm_eff_mls_threshold
    ],

    "External Error Detection Rate": [
        d1_eff_msr_detection,
        d1_eff_mls_detection,
        d2_eff_msr_detection,
        d2_eff_mls_detection
    ],

    "External Coverage": [
        d1_eff_msr_coverage,
        d1_eff_mls_coverage,
        d2_eff_msr_coverage,
        d2_eff_mls_coverage
    ],

    "External Silent Failures": [
        d1_eff_msr_silent,
        d1_eff_mls_silent,
        d2_eff_msr_silent,
        d2_eff_mls_silent
    ]
})


print("\n" + "=" * 70)
print("EFFICIENTNET-B0 FAILURE DETECTION SUMMARY")
print("=" * 70)

display(
    efficientnet_failure_summary.round(4)
)


Processing BDNeuro -> PMRAM...
Processing PMRAM -> BDNeuro...

EFFICIENTNET-B0 FAILURE DETECTION SUMMARY


,Direction,Method,Internal AUROC-F,External AUROC-F,Source Threshold,External Error Detection Rate,External Coverage,External Silent Failures
0,BDNeuro -> PMRAM,MSR,0.9586,0.9341,0.9725,0.9123,0.7631,5
1,BDNeuro -> PMRAM,MLS,0.9417,0.9020,4.5478,0.9649,0.6433,2
2,PMRAM -> BDNeuro,MSR,0.8603,0.8536,0.9288,0.8372,0.5622,326
3,PMRAM -> BDNeuro,MLS,0.8548,0.8576,3.4615,0.9096,0.4809,181


## 3. Failure Detection with MC-Dropout

MC-Dropout is used to estimate predictive uncertainty through repeated stochastic forward passes. Predictive entropy is used as the failure score, with thresholds selected from source validation data.




In [9]:
# ============================================================
# EfficientNet-B0 + Dropout
# MC Dropout Model
# Direction 1: BDNeuro -> PMRAM
# ============================================================

import torch
import torch.nn as nn
from torchvision import models
import time
import copy


# ------------------------------------------------------------
# 1. Create NEW EfficientNet-B0
# ------------------------------------------------------------

model_bd_eff_mcd = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.DEFAULT
)

in_features = model_bd_eff_mcd.classifier[1].in_features

# EfficientNet classifier:
# Dropout(p=0.2) + Linear
model_bd_eff_mcd.classifier[1] = nn.Linear(
    in_features,
    4
)

model_bd_eff_mcd = model_bd_eff_mcd.to(device)


print("EfficientNet-B0 classifier:")
print(model_bd_eff_mcd.classifier)


# ------------------------------------------------------------
# 2. Check Dropout
# ------------------------------------------------------------

dropout_count = sum(
    isinstance(module, nn.Dropout)
    for module in model_bd_eff_mcd.modules()
)

print("\nNumber of Dropout layers:", dropout_count)


# ------------------------------------------------------------
# 3. Loss + optimizer
# ------------------------------------------------------------

criterion_bd_eff_mcd = nn.CrossEntropyLoss()

optimizer_bd_eff_mcd = torch.optim.Adam(
    model_bd_eff_mcd.parameters(),
    lr=0.0001
)


# ------------------------------------------------------------
# 4. Train for 5 epochs
# Keep BEST validation model
# ------------------------------------------------------------

num_epochs = 5

best_val_accuracy = 0.0

best_model_weights = copy.deepcopy(
    model_bd_eff_mcd.state_dict()
)

training_start = time.time()

print("\nTraining EfficientNet-B0 + Dropout")
print("Direction: BDNeuro -> PMRAM")
print("=" * 65)


for epoch in range(num_epochs):

    epoch_start = time.time()

    # ---------------- TRAIN ----------------

    model_bd_eff_mcd.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in bd_train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_bd_eff_mcd.zero_grad()

        outputs = model_bd_eff_mcd(images)

        loss = criterion_bd_eff_mcd(
            outputs,
            labels
        )

        loss.backward()
        optimizer_bd_eff_mcd.step()

        train_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(dim=1)

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)


    train_loss /= train_total
    train_accuracy = train_correct / train_total


    # ---------------- VALIDATION ----------------

    model_bd_eff_mcd.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in bd_val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_bd_eff_mcd(images)

            loss = criterion_bd_eff_mcd(
                outputs,
                labels
            )

            val_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)


    val_loss /= val_total
    val_accuracy = val_correct / val_total


    # ---------------- BEST MODEL ----------------

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        best_model_weights = copy.deepcopy(
            model_bd_eff_mcd.state_dict()
        )


    epoch_minutes = (
        time.time() - epoch_start
    ) / 60


    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Time: {epoch_minutes:.1f} min"
    )


# ------------------------------------------------------------
# 5. Restore best model
# ------------------------------------------------------------

model_bd_eff_mcd.load_state_dict(
    best_model_weights
)


total_minutes = (
    time.time() - training_start
) / 60


print("\n" + "=" * 65)
print("TRAINING COMPLETE")

print(
    f"Best Validation Accuracy: "
    f"{best_val_accuracy:.4f}"
)

print(
    f"Total Training Time: "
    f"{total_minutes:.1f} minutes"
)


# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------

torch.save(
    model_bd_eff_mcd.state_dict(),
    "efficientnet_b0_bdneuro_mcd_best.pth"
)

print(
    "\nModel saved as:"
    "\nefficientnet_b0_bdneuro_mcd_best.pth"
)


EfficientNet-B0 classifier:
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=4, bias=True)
)

Number of Dropout layers: 1

Training EfficientNet-B0 + Dropout
Direction: BDNeuro -> PMRAM
Epoch 1/5 | Train Loss: 0.6418 | Train Acc: 0.7822 | Val Loss: 0.3105 | Val Acc: 0.9025 | Time: 18.9 min
Epoch 2/5 | Train Loss: 0.2518 | Train Acc: 0.9099 | Val Loss: 0.1623 | Val Acc: 0.9439 | Time: 15.4 min
Epoch 3/5 | Train Loss: 0.1486 | Train Acc: 0.9512 | Val Loss: 0.1170 | Val Acc: 0.9585 | Time: 16.2 min
Epoch 4/5 | Train Loss: 0.0887 | Train Acc: 0.9709 | Val Loss: 0.0984 | Val Acc: 0.9630 | Time: 15.4 min
Epoch 5/5 | Train Loss: 0.0698 | Train Acc: 0.9772 | Val Loss: 0.1042 | Val Acc: 0.9720 | Time: 15.3 min

TRAINING COMPLETE
Best Validation Accuracy: 0.9720
Total Training Time: 81.2 minutes

Model saved as:
efficientnet_b0_bdneuro_mcd_best.pth


In [10]:
# ============================================================
# Baseline Evaluation
# EfficientNet-B0 + Dropout
# BDNeuro -> PMRAM
# ============================================================

from sklearn.metrics import accuracy_score, classification_report

def evaluate_model(model, data_loader, device):

    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    report = classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Glioma",
            "Meningioma",
            "No Tumor",
            "Pituitary"
        ],
        digits=4
    )

    return accuracy, report


# Internal: BDNeuro
bd_eff_mcd_test_accuracy, bd_eff_mcd_test_report = evaluate_model(
    model_bd_eff_mcd,
    bd_test_loader,
    device
)

print("=" * 60)
print("BDNeuro INTERNAL TEST")
print("=" * 60)

print(
    "Accuracy:",
    bd_eff_mcd_test_accuracy
)

print("\nClassification Report:")
print(bd_eff_mcd_test_report)


# External: PMRAM
pm_eff_mcd_external_accuracy, pm_eff_mcd_external_report = evaluate_model(
    model_bd_eff_mcd,
    pm_external_loader,
    device
)

print("\n" + "=" * 60)
print("PMRAM EXTERNAL TEST")
print("=" * 60)

print(
    "Accuracy:",
    pm_eff_mcd_external_accuracy
)

print("\nClassification Report:")
print(pm_eff_mcd_external_report)


BDNeuro INTERNAL TEST
Accuracy: 0.9741282339707537

Classification Report:
              precision    recall  f1-score   support

      Glioma     0.9815    0.9695    0.9755       328
  Meningioma     0.9769    0.9548    0.9657       221
    No Tumor     0.9219    1.0000    0.9593       118
   Pituitary     0.9910    0.9865    0.9887       222

    accuracy                         0.9741       889
   macro avg     0.9678    0.9777    0.9723       889
weighted avg     0.9748    0.9741    0.9742       889


PMRAM EXTERNAL TEST
Accuracy: 0.9340425531914893

Classification Report:
              precision    recall  f1-score   support

      Glioma     0.9858    0.9405    0.9627       370
  Meningioma     0.9632    0.8012    0.8748       327
    No Tumor     0.8927    1.0000    0.9433       341
   Pituitary     0.9082    0.9839    0.9445       372

    accuracy                         0.9340      1410
   macro avg     0.9375    0.9314    0.9313      1410
weighted avg     0.9376    0.9340   

In [ ]:
# ============================================================
# RESTORE MC-DROPOUT RESULTS
# Direction: BDNeuro -> PMRAM
# No training
# ============================================================

import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# Turn on Dropout while keeping BatchNorm in evaluation mode
# ------------------------------------------------------------

def enable_mc_dropout(model):

    model.eval()

    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.train()


# ------------------------------------------------------------
# MC-Dropout inference
# ------------------------------------------------------------

def collect_mc_entropy_results(
    model,
    loader,
    device,
    mc_passes=20
):

    enable_mc_dropout(model)

    all_entropy = []
    all_correct = []
    all_confidence = []
    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            mc_probabilities = []

            for _ in range(mc_passes):

                outputs = model(images)

                probabilities = F.softmax(
                    outputs,
                    dim=1
                )

                mc_probabilities.append(
                    probabilities.unsqueeze(0)
                )

            mc_probabilities = torch.cat(
                mc_probabilities,
                dim=0
            )

            mean_probabilities = (
                mc_probabilities.mean(dim=0)
            )

            entropy = -torch.sum(
                mean_probabilities *
                torch.log(
                    mean_probabilities + 1e-12
                ),
                dim=1
            )

            confidence, predictions = (
                mean_probabilities.max(dim=1)
            )

            correct = (
                predictions == labels
            ).long()

            all_entropy.extend(
                entropy.cpu().numpy()
            )

            all_correct.extend(
                correct.cpu().numpy()
            )

            all_confidence.extend(
                confidence.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    return pd.DataFrame({
        "entropy": all_entropy,
        "confidence": all_confidence,
        "correct": all_correct,
        "label": all_labels,
        "prediction": all_predictions
    })


# ------------------------------------------------------------
# 1. BDNeuro Validation
# Used ONLY to determine threshold
# ------------------------------------------------------------

print("1/3 Running MC-Dropout on BDNeuro VALIDATION...")

bd_eff_val_mc_entropy_results = (
    collect_mc_entropy_results(
        model_bd_eff_mcd,
        bd_val_loader,
        device,
        mc_passes=20
    )
)

print(
    "Finished:",
    len(bd_eff_val_mc_entropy_results),
    "predictions"
)


# ------------------------------------------------------------
# 2. BDNeuro Internal Test
# ------------------------------------------------------------

print(
    "\n2/3 Running MC-Dropout on BDNeuro INTERNAL TEST..."
)

bd_eff_internal_mc_entropy_results = (
    collect_mc_entropy_results(
        model_bd_eff_mcd,
        bd_test_loader,
        device,
        mc_passes=20
    )
)

bd_internal_mc_auroc_f = roc_auc_score(
    1 - bd_eff_internal_mc_entropy_results["correct"],
    bd_eff_internal_mc_entropy_results["entropy"]
)

print(
    "BDNeuro Internal MC-Dropout AUROC-F:",
    round(bd_internal_mc_auroc_f, 4)
)


# ------------------------------------------------------------
# 3. PMRAM External
# ------------------------------------------------------------

print(
    "\n3/3 Running MC-Dropout on PMRAM EXTERNAL..."
)

pm_eff_mc_entropy_results = (
    collect_mc_entropy_results(
        model_bd_eff_mcd,
        pm_external_loader,
        device,
        mc_passes=20
    )
)

pm_external_mc_auroc_f = roc_auc_score(
    1 - pm_eff_mc_entropy_results["correct"],
    pm_eff_mc_entropy_results["entropy"]
)

print(
    "PMRAM External MC-Dropout AUROC-F:",
    round(pm_external_mc_auroc_f, 4)
)


print("\n====================================")
print("MC-DROPOUT RESTORE COMPLETE")
print("====================================")


In [5]:
# ============================================================
# MC-DROPOUT THRESHOLD + FAILURE DETECTION
# Direction: BDNeuro -> PMRAM
# Threshold is selected ONLY from BDNeuro validation
# ============================================================

from sklearn.metrics import roc_curve
import numpy as np


# ------------------------------------------------------------
# 1. Find threshold using SOURCE VALIDATION only
# ------------------------------------------------------------

def find_entropy_threshold(results):

    y_error = 1 - results["correct"].values
    scores = results["entropy"].values

    fpr, tpr, thresholds = roc_curve(
        y_error,
        scores
    )

    # Youden's J statistic
    j_scores = tpr - fpr
    best_index = np.argmax(j_scores)

    return thresholds[best_index]


bd_eff_entropy_threshold = find_entropy_threshold(
    bd_eff_val_mc_entropy_results
)

print(
    "BDNeuro Validation Entropy Threshold:",
    round(bd_eff_entropy_threshold, 4)
)


# ------------------------------------------------------------
# 2. Apply SAME threshold to PMRAM external dataset
# ------------------------------------------------------------

pm_entropy = pm_eff_mc_entropy_results["entropy"].values
pm_correct = pm_eff_mc_entropy_results["correct"].values

# Flag prediction as uncertain / possible failure
pm_flagged = (
    pm_entropy >= bd_eff_entropy_threshold
)

# Actual errors
pm_errors = (
    pm_correct == 0
)

total_errors = pm_errors.sum()

detected_errors = (
    pm_flagged & pm_errors
).sum()

silent_failures = (
    (~pm_flagged) & pm_errors
).sum()

total_flagged = pm_flagged.sum()


# Error Detection Rate
error_detection_rate = (
    detected_errors / total_errors
    if total_errors > 0
    else np.nan
)

# Coverage = proportion NOT flagged
coverage = (
    (~pm_flagged).mean()
)


# ------------------------------------------------------------
# 3. Print results
# ------------------------------------------------------------

print("\n========================================")
print("BDNeuro -> PMRAM")
print("MC-DROPOUT FAILURE DETECTION")
print("========================================")

print("External images:", len(pm_correct))
print("External errors:", int(total_errors))

print(
    "Detected errors:",
    int(detected_errors)
)

print(
    "Silent failures:",
    int(silent_failures)
)

print(
    "Total flagged:",
    int(total_flagged)
)

print(
    "Error Detection Rate:",
    round(error_detection_rate, 4)
)

print(
    "Coverage:",
    round(coverage, 4)
)

print(
    "External AUROC-F:",
    round(pm_external_mc_auroc_f, 4)
)

print("========================================")


BDNeuro Validation Entropy Threshold: 0.2487

BDNeuro -> PMRAM
MC-DROPOUT FAILURE DETECTION
External images: 1410
External errors: 94
Detected errors: 80
Silent failures: 14
Total flagged: 241
Error Detection Rate: 0.8511
Coverage: 0.8291
External AUROC-F: 0.9365


In [6]:
# ============================================================
# PREPARE PMRAM MODEL FOR MC-DROPOUT
# Direction: PMRAM -> BDNeuro
# NO TRAINING
# ============================================================

import copy
import torch

# Make an independent copy of the already-trained PMRAM model
model_pm_mcd = copy.deepcopy(model_pm_eff)

model_pm_mcd = model_pm_mcd.to(device)

# Check dropout layers
dropout_layers = []

for name, module in model_pm_mcd.named_modules():
    if isinstance(module, torch.nn.Dropout):
        dropout_layers.append((name, module.p))

print("PMRAM EfficientNet MC-Dropout model ready.")
print("Number of Dropout layers:", len(dropout_layers))
print("Dropout layers:", dropout_layers)

print("\nNo training performed.")
print("Ready for PMRAM -> BDNeuro MC-Dropout.")


PMRAM EfficientNet MC-Dropout model ready.
Number of Dropout layers: 1
Dropout layers: [('classifier.0', 0.2)]

No training performed.
Ready for PMRAM -> BDNeuro MC-Dropout.


In [9]:
# ============================================================
# PMRAM -> BDNeuro
# MC-DROPOUT ON PMRAM VALIDATION
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F


# ------------------------------------------------------------
# Activate Dropout during inference
# ------------------------------------------------------------

def enable_mc_dropout(model):

    model.eval()

    for module in model.modules():

        if isinstance(module, torch.nn.Dropout):
            module.train()


# ------------------------------------------------------------
# Collect MC-Dropout predictive entropy
# ------------------------------------------------------------

def collect_mc_entropy_results(
    model,
    loader,
    device,
    mc_passes=20
):

    enable_mc_dropout(model)

    all_labels = []
    all_preds = []
    all_entropy = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            mc_probs = []

            # Multiple stochastic forward passes
            for _ in range(mc_passes):

                outputs = model(images)

                probs = F.softmax(
                    outputs,
                    dim=1
                )

                mc_probs.append(
                    probs.unsqueeze(0)
                )

            # Shape:
            # [MC passes, batch, classes]

            mc_probs = torch.cat(
                mc_probs,
                dim=0
            )

            # Mean probability across MC passes

            mean_probs = mc_probs.mean(
                dim=0
            )

            # Final prediction

            preds = mean_probs.argmax(
                dim=1
            )

            # Predictive entropy

            entropy = -(
                mean_probs *
                torch.log(
                    mean_probs + 1e-12
                )
            ).sum(dim=1)

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_entropy.extend(
                entropy.cpu().numpy()
            )


    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    results = pd.DataFrame({

        "label": all_labels,

        "prediction": all_preds,

        "entropy": all_entropy

    })


    results["correct"] = (

        results["label"]
        ==
        results["prediction"]

    ).astype(int)


    return results


# ============================================================
# RUN ON PMRAM VALIDATION
# ============================================================

print(
    "Running MC-Dropout on PMRAM VALIDATION..."
)


pm_eff_val_mc_entropy_results = (
    collect_mc_entropy_results(

        model_pm_mcd, # correct model name

        pm_val_loader,

        device,

        mc_passes=20
    )
)


# ============================================================
# RESULTS
# ============================================================

print(
    "\nFinished:",
    len(pm_eff_val_mc_entropy_results),
    "predictions"
)


validation_errors = (

    pm_eff_val_mc_entropy_results[
        "correct"
    ] == 0

).sum()


print(
    "Validation errors:",
    validation_errors
)


correct_entropy = (

    pm_eff_val_mc_entropy_results[

        pm_eff_val_mc_entropy_results[
            "correct"
        ] == 1

    ]["entropy"].mean()

)


incorrect_entropy = (

    pm_eff_val_mc_entropy_results[

        pm_eff_val_mc_entropy_results[
            "correct"
        ] == 0

    ]["entropy"].mean()

)


print(
    "Average entropy - correct:",
    correct_entropy
)


print(
    "Average entropy - incorrect:",
    incorrect_entropy
)


print(
    "\n========================================"
)

print(
    "PMRAM VALIDATION MC-DROPOUT COMPLETE"
)

print(
    "========================================"
)


Running MC-Dropout on PMRAM VALIDATION...

Finished: 211 predictions
Validation errors: 4
Average entropy - correct: 0.16661727
Average entropy - incorrect: 0.5114491

PMRAM VALIDATION MC-DROPOUT COMPLETE


In [10]:
# ============================================================
# PMRAM -> BDNeuro
# FIND MC-DROPOUT ENTROPY THRESHOLD FROM PMRAM VALIDATION
# ============================================================

import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score


def find_entropy_threshold(results):

    # 1 = prediction failure
    failure_labels = 1 - results["correct"].values

    entropy_scores = results["entropy"].values

    # AUROC-F
    auroc_f = roc_auc_score(
        failure_labels,
        entropy_scores
    )

    # ROC curve
    fpr, tpr, thresholds = roc_curve(
        failure_labels,
        entropy_scores
    )

    # Youden's J statistic
    j_scores = tpr - fpr

    best_index = np.argmax(j_scores)

    best_threshold = thresholds[best_index]

    return best_threshold, auroc_f


# ------------------------------------------------------------
# Calculate threshold ONLY from PMRAM validation
# ------------------------------------------------------------

pm_eff_entropy_threshold, pm_eff_val_mc_auroc_f = (
    find_entropy_threshold(
        pm_eff_val_mc_entropy_results
    )
)


print(
    "========================================"
)

print(
    "PMRAM -> BDNeuro"
)

print(
    "VALIDATION THRESHOLD SELECTION"
)

print(
    "========================================"
)

print(
    "PMRAM Validation MC-Dropout AUROC-F:",
    round(pm_eff_val_mc_auroc_f, 4)
)

print(
    "PMRAM Validation Entropy Threshold:",
    round(pm_eff_entropy_threshold, 4)
)

print(
    "Validation errors:",
    (
        pm_eff_val_mc_entropy_results["correct"] == 0
    ).sum()
)

print(
    "========================================"
)


PMRAM -> BDNeuro
VALIDATION THRESHOLD SELECTION
PMRAM Validation MC-Dropout AUROC-F: 0.8853
PMRAM Validation Entropy Threshold: 0.2921
Validation errors: 4


In [11]:
# ============================================================
# PMRAM -> BDNeuro
# MC-DROPOUT EXTERNAL FAILURE DETECTION
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 1. Run MC-Dropout on BDNeuro External Dataset
# ------------------------------------------------------------

print("========================================")
print("PMRAM -> BDNeuro")
print("Running MC-Dropout on BDNeuro EXTERNAL...")
print("This may take some time on CPU.")
print("========================================")


bd_eff_external_mc_results = collect_mc_entropy_results(
    model_pm_mcd,
    bd_external_loader,
    device,
    mc_passes=20
)


print(
    "\nFinished:",
    len(bd_eff_external_mc_results),
    "external predictions"
)


# ------------------------------------------------------------
# 2. External AUROC-F
# ------------------------------------------------------------

external_failure_labels = (
    1 - bd_eff_external_mc_results["correct"].values
)

external_entropy_scores = (
    bd_eff_external_mc_results["entropy"].values
)


pm_to_bd_mc_auroc_f = roc_auc_score(
    external_failure_labels,
    external_entropy_scores
)


# ------------------------------------------------------------
# 3. Apply FIXED threshold from PMRAM Validation
# ------------------------------------------------------------

threshold = pm_eff_entropy_threshold


bd_eff_external_mc_results["flagged"] = (
    bd_eff_external_mc_results["entropy"]
    >= threshold
).astype(int)


# ------------------------------------------------------------
# 4. Calculate failure-detection statistics
# ------------------------------------------------------------

total_images = len(
    bd_eff_external_mc_results
)

total_errors = (
    bd_eff_external_mc_results["correct"] == 0
).sum()


detected_errors = (
    (
        bd_eff_external_mc_results["correct"] == 0
    )
    &
    (
        bd_eff_external_mc_results["flagged"] == 1
    )
).sum()


silent_failures = (
    (
        bd_eff_external_mc_results["correct"] == 0
    )
    &
    (
        bd_eff_external_mc_results["flagged"] == 0
    )
).sum()


total_flagged = (
    bd_eff_external_mc_results["flagged"] == 1
).sum()


accepted_images = (
    bd_eff_external_mc_results["flagged"] == 0
).sum()


# Error Detection Rate

if total_errors > 0:
    error_detection_rate = (
        detected_errors / total_errors
    )
else:
    error_detection_rate = np.nan


# Coverage = fraction NOT flagged

coverage = (
    accepted_images / total_images
)


# ------------------------------------------------------------
# 5. Print final results
# ------------------------------------------------------------

print("\n")
print("========================================")
print("PMRAM -> BDNeuro")
print("MC-DROPOUT EXTERNAL FAILURE DETECTION")
print("========================================")

print(
    "Validation-derived threshold:",
    round(threshold, 4)
)

print(
    "External images:",
    total_images
)

print(
    "External errors:",
    total_errors
)

print(
    "Detected errors:",
    detected_errors
)

print(
    "Silent failures:",
    silent_failures
)

print(
    "Total flagged:",
    total_flagged
)

print(
    "Error Detection Rate:",
    round(error_detection_rate, 4)
)

print(
    "Coverage:",
    round(coverage, 4)
)

print(
    "External AUROC-F:",
    round(pm_to_bd_mc_auroc_f, 4)
)

print("========================================")


PMRAM -> BDNeuro
Running MC-Dropout on BDNeuro EXTERNAL...
This may take some time on CPU.

Finished: 5941 external predictions


PMRAM -> BDNeuro
MC-DROPOUT EXTERNAL FAILURE DETECTION
Validation-derived threshold: 0.2921
External images: 5941
External errors: 2010
Detected errors: 1710
Silent failures: 300
Total flagged: 2684
Error Detection Rate: 0.8507
Coverage: 0.5482
External AUROC-F: 0.8595


In [14]:
# ============================================================
# FIX FINAL EFFICIENTNET-B0 MC-DROPOUT TABLE
# Correct mapping of external datasets
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score


def make_mc_summary(results, threshold, direction):

    correct = np.asarray(results["correct"]).astype(int)
    entropy = np.asarray(results["entropy"]).astype(float)

    errors = (correct == 0)
    flagged = (entropy >= threshold)

    total_images = len(correct)
    total_errors = int(errors.sum())
    detected_errors = int((errors & flagged).sum())
    silent_failures = int((errors & ~flagged).sum())
    total_flagged = int(flagged.sum())

    error_detection_rate = (
        detected_errors / total_errors
        if total_errors > 0 else np.nan
    )

    coverage = 1 - (total_flagged / total_images)

    auroc_f = roc_auc_score(
        errors.astype(int),
        entropy
    )

    return {
        "Direction": direction,
        "Method": "MC-Dropout",
        "Threshold": threshold,
        "External Images": total_images,
        "External Errors": total_errors,
        "Detected Errors": detected_errors,
        "Silent Failures": silent_failures,
        "Error Detection Rate": error_detection_rate,
        "Coverage": coverage,
        "External AUROC-F": auroc_f
    }


# ------------------------------------------------------------
# BDNeuro -> PMRAM
# External dataset MUST contain 1410 images
# ------------------------------------------------------------

bd_to_pm = make_mc_summary(
    pm_eff_mc_entropy_results,
    bd_eff_entropy_threshold,
    "BDNeuro -> PMRAM"
)


# ------------------------------------------------------------
# PMRAM -> BDNeuro
# External dataset MUST contain 5941 images
# ------------------------------------------------------------

pm_to_bd = make_mc_summary(
    bd_eff_external_mc_results,
    pm_eff_entropy_threshold,
    "PMRAM -> BDNeuro"
)


# ------------------------------------------------------------
# Create final table
# ------------------------------------------------------------

efficientnet_mc_final_table = pd.DataFrame([
    bd_to_pm,
    pm_to_bd
])


# Safety check
assert efficientnet_mc_final_table.loc[
    efficientnet_mc_final_table["Direction"] == "BDNeuro -> PMRAM",
    "External Images"
].iloc[0] == 1410

assert efficientnet_mc_final_table.loc[
    efficientnet_mc_final_table["Direction"] == "PMRAM -> BDNeuro",
    "External Images"
].iloc[0] == 5941


# Display rounded version
display_table = efficientnet_mc_final_table.copy()

for col in [
    "Threshold",
    "Error Detection Rate",
    "Coverage",
    "External AUROC-F"
]:
    display_table[col] = display_table[col].round(4)


print("=" * 100)
print("FINAL CORRECTED EFFICIENTNET-B0 MC-DROPOUT TABLE")
print("=" * 100)

display(display_table)


# Save full precision
efficientnet_mc_final_table.to_csv(
    "efficientnet_b0_mc_dropout_final_results_CORRECTED.csv",
    index=False
)

print("\nCORRECTED TABLE SAVED SUCCESSFULLY")
print("efficientnet_b0_mc_dropout_final_results_CORRECTED.csv")


FINAL CORRECTED EFFICIENTNET-B0 MC-DROPOUT TABLE


,Direction,Method,Threshold,External Images,External Errors,Detected Errors,Silent Failures,Error Detection Rate,Coverage,External AUROC-F
0,BDNeuro -> PMRAM,MC-Dropout,0.2487,1410,94,80,14,0.8511,0.8291,0.9365
1,PMRAM -> BDNeuro,MC-Dropout,0.2921,5941,2010,1710,300,0.8507,0.5482,0.8595



CORRECTED TABLE SAVED SUCCESSFULLY
efficientnet_b0_mc_dropout_final_results_CORRECTED.csv


## 4. Key Findings

- EfficientNet-B0 showed strong internal performance on both datasets.
- Cross-dataset performance was strongly direction-dependent.
- BDNeuro → PMRAM generalized substantially better than PMRAM → BDNeuro.
- The largest class-specific degradation in the difficult direction occurred for meningioma.
- MSR, MLS, and MC-Dropout were able to identify a substantial proportion of external prediction failures, although error detection involved a trade-off with coverage.

These results motivate further analysis of dataset shift and class-specific failure patterns.

